Mi idea es usar unsloth, un modelo bueno/instruct y actual como gemma, hacerle finetuning y luego agregar buenos prompts

### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 8000 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.4: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `model.base_model.model.model` require gradients


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Alpaca.ipynb)

For text completions like novel writing, try this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Mistral_(7B)-Text_Completion.ipynb).

### **Converti los datos de entrenamiento a formato alpaca y lo subi a mi hugginface**

In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

from datasets import load_dataset
dataset = load_dataset("andrewmos/indian-legal-summaries-alpaca-format", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

README.md:   0%|          | 0.00/482 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/24.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.50M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/960 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

## Inference de modelo sin finetunear

In [ ]:
# alpaca_prompt = Copied from above
INSTRUCTION = "Provide a concise and accurate summary of the following legal judgment. Focus on the key facts, the legal reasoning, and the final verdict."
input_text = "Leave granted.\nHeard the learned counsel senior counsel appearing for the\nappellant and the learned ASG appearing for the first\nrespondent/Directorate of Enforcement.\nThe present appellant has been arrested on 10th March, 2023 in\nconnection with the offence punishable under Section 3 of the\nPrevention of Money-laundering Act, 2002 (for short, “the PMLA”).\nAfter the submissions are heard, the learned ASG has fairly\nleft it to the Court to decide the prayer for grant of bail to the\nappellant.\nEven otherwise, we find that the appellant is entitled to be\nenlarged on bail in accordance with Section 45(1)(ii) of the PMLA\non appropriate terms and conditions, till the disposal of the\ncomplaint case filed by the first respondent/Directorate of\nEnforcement under the PMLA. In view of the fair stand taken by the\nlearned ASG, we are not recording detailed reasons.\nFor that purpose, we direct that the appellant shall be\nproduced before the Special Court within a period of one week from\ntoday. The Special Court shall enlarge the appellant on bail on\nappropriate terms and conditions, till the trial of the complaint\ncase concludes.\nThe Appeal is, accordingly, allowed.\nPetition(s) for Special Leave to Appeal (Crl.) No(s). 16236/2023\n(Arising out of impugned final judgment and order dated 06-12-2023\nin BA No. 3233/2023 passed by the High Court of Judicature at\nBombay)\nDate : 12-02-2024 This matter was called on for hearing today.\nFor Petitioner(s) Mr. Kapil Sibal, Sr. Adv.\nMr. Devadatt Kamat, Sr. Adv.\nMr. Rohit Sharma, Adv.\nMr. Sunny Jain, Adv.\nMr. Rajesh Inamdar, Adv.\nMr. Nikhil Purohit, Adv.\nMr. Jatin Lalwani, Adv.\nMr. Shardul Singh, Adv.\nMs. Prerna Gandhi, Adv.\nMr. Anish Sahapurkar, Adv.\nMs. Aparajita Jamwal, Adv.\nMr. Abhik Chimney, Adv.\nMr. Revanta Solanki, Adv.\nMr. Kumar Dushyant Singh, AOR\nFor Respondent(s) Mr. Suryaprakash V Raju, A.S.G.\nMr. Mukesh Kumar Maroria, AOR\nMr. Annam Venkatesh, Adv.\nMr. Zoheb Hussain, Adv.\nMr. Rajat Nair, Adv.\nMs. Yugandhara Pawar Jha, Adv.\nMr. Siddharth Dharmadhikari, Adv.\nMr. Aaditya Aniruddha Pande, AOR\nMr. Bharat Bagla, Adv.\nMr. Sourav Singh, Adv.\nMr. Aditya Krishna, Adv.\nMs. Preet S. Phanse, Adv.\nUPON hearing the counsel the Court made the following\nLeave granted.\nThe Appeal is allowed in terms of the signed order. The\noperative portion of the order reads thus:\n“For that purpose, we direct that the appellant\nshall be produced before the Special Court within a\nperiod of one week from today. The Special Court\nshall enlarge the appellant on bail on appropriate\nterms and conditions, till the trial of the\ncomplaint case concludes.\nThe Appeal is, accordingly, allowed.”\nPending applications stand disposed of accordingly."

FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        INSTRUCTION, # instruction
        input_text, # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048)

<bos>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Provide a concise and accurate summary of the following legal judgment. Focus on the key facts, the legal reasoning, and the final verdict.

### Input:
Leave granted.
Heard the learned counsel senior counsel appearing for the
appellant and the learned ASG appearing for the first
respondent/Directorate of Enforcement.
The present appellant has been arrested on 10th March, 2023 in
connection with the offence punishable under Section 3 of the
Prevention of Money-laundering Act, 2002 (for short, “the PMLA”).
After the submissions are heard, the learned ASG has fairly
left it to the Court to decide the prayer for grant of bail to the
appellant.
Even otherwise, we find that the appellant is entitled to be
enlarged on bail in accordance with Section 45(1)(ii) of the PMLA
on appropriate terms and conditions, till th

## Train
<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [5]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        #max_steps = 50,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/960 [00:00<?, ? examples/s]

In [6]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
1.512 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 960 | Num Epochs = 1 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,045,760 of 1,012,931,712 (1.29% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.711200
2,2.963500
3,2.760800
4,2.834600
5,2.630400
6,2.673600
7,2.616200
8,2.489000
9,2.425800
10,2.697900


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Probar modelo finetuneado

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
INSTRUCTION = "Provide a concise and accurate summary of the following legal judgment. Focus on the key facts, the legal reasoning, and the final verdict."
input_text = "Leave granted.\nHeard the learned counsel senior counsel appearing for the\nappellant and the learned ASG appearing for the first\nrespondent/Directorate of Enforcement.\nThe present appellant has been arrested on 10th March, 2023 in\nconnection with the offence punishable under Section 3 of the\nPrevention of Money-laundering Act, 2002 (for short, “the PMLA”).\nAfter the submissions are heard, the learned ASG has fairly\nleft it to the Court to decide the prayer for grant of bail to the\nappellant.\nEven otherwise, we find that the appellant is entitled to be\nenlarged on bail in accordance with Section 45(1)(ii) of the PMLA\non appropriate terms and conditions, till the disposal of the\ncomplaint case filed by the first respondent/Directorate of\nEnforcement under the PMLA. In view of the fair stand taken by the\nlearned ASG, we are not recording detailed reasons.\nFor that purpose, we direct that the appellant shall be\nproduced before the Special Court within a period of one week from\ntoday. The Special Court shall enlarge the appellant on bail on\nappropriate terms and conditions, till the trial of the complaint\ncase concludes.\nThe Appeal is, accordingly, allowed.\nPetition(s) for Special Leave to Appeal (Crl.) No(s). 16236/2023\n(Arising out of impugned final judgment and order dated 06-12-2023\nin BA No. 3233/2023 passed by the High Court of Judicature at\nBombay)\nDate : 12-02-2024 This matter was called on for hearing today.\nFor Petitioner(s) Mr. Kapil Sibal, Sr. Adv.\nMr. Devadatt Kamat, Sr. Adv.\nMr. Rohit Sharma, Adv.\nMr. Sunny Jain, Adv.\nMr. Rajesh Inamdar, Adv.\nMr. Nikhil Purohit, Adv.\nMr. Jatin Lalwani, Adv.\nMr. Shardul Singh, Adv.\nMs. Prerna Gandhi, Adv.\nMr. Anish Sahapurkar, Adv.\nMs. Aparajita Jamwal, Adv.\nMr. Abhik Chimney, Adv.\nMr. Revanta Solanki, Adv.\nMr. Kumar Dushyant Singh, AOR\nFor Respondent(s) Mr. Suryaprakash V Raju, A.S.G.\nMr. Mukesh Kumar Maroria, AOR\nMr. Annam Venkatesh, Adv.\nMr. Zoheb Hussain, Adv.\nMr. Rajat Nair, Adv.\nMs. Yugandhara Pawar Jha, Adv.\nMr. Siddharth Dharmadhikari, Adv.\nMr. Aaditya Aniruddha Pande, AOR\nMr. Bharat Bagla, Adv.\nMr. Sourav Singh, Adv.\nMr. Aditya Krishna, Adv.\nMs. Preet S. Phanse, Adv.\nUPON hearing the counsel the Court made the following\nLeave granted.\nThe Appeal is allowed in terms of the signed order. The\noperative portion of the order reads thus:\n“For that purpose, we direct that the appellant\nshall be produced before the Special Court within a\nperiod of one week from today. The Special Court\nshall enlarge the appellant on bail on appropriate\nterms and conditions, till the trial of the\ncomplaint case concludes.\nThe Appeal is, accordingly, allowed.”\nPending applications stand disposed of accordingly."

FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        INSTRUCTION, # instruction
        input_text, # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 4096)

<bos>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Convert the following purchase order text to a structured JSON format. Extract buyer information (name, email, contact details, addresses), purchases (product names, quantities, currencies, discount codes), and shipping preferences (method, preferred delivery date).

### Input:
¡Hola! Quería hacer un pedido. Necesito 5 botellas de vino tinto y 3 cajas de galletas. Mi email es juan@example.com. Por favor envíenlo a mi dirección en Madrid.

### Response:
{"buyer":{"name":null,"email": juan@example.com,"contact":null,"addresses":null,"preferencias":null},"purchases":null,"shipping":{"method":"Delivery ","preferred_by":null,"preferred_by":null},"currencies":null}<end_of_turn>


## Guardar modelo en hugginface

In [ ]:
# Merge to 16bit
if True: model.push_to_hub_merged("andrewmos/indian-legal-summaries-finetuned", tokenizer, save_method = "merged_16bit", token = "")